# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tanay1122/flyrank-ml/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

Look before deciding (the skill's order of operations). Every field this audit tests gets a distribution read FIRST — with the heavy-tail plan: web traffic metrics are giants-plus-a-long-tail, so plain Pearson on raw values is giant-dominated; verdicts below default to buckets (grouped rates), Spearman rank correlation, and `log1p` where a continuous read is needed. Trap audits at the bottom: (a) the average of per-page CTRs vs the TRUE aggregate rate (clicks/impressions weighted), (b) `avg_position = 0` semantics — zero means "no data" in the starter dataset, so the release's positional column gets a zero-count receipt before any test leans on it.

Same slice as the lane's baseline: month=2026-03, one row = one page, kept pages pass the 100-imp / 10-day floor, row-days filtered `gsc_data_available IS TRUE`, aggregates ≤ 2026-03-20 only. Labels (03-21..31) are rulers, never inputs.

In [1]:
# This cell is for CODE (numbers, a query, a check).
%pip -q install duckdb huggingface_hub scikit-learn
import os
os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")

from pathlib import Path
import subprocess
import warnings
warnings.filterwarnings("ignore", message="IProgress not found")  # tqdm warns when jupyter widgets are absent

import duckdb

print("duckdb", duckdb.__version__)

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass
if not HF_TOKEN:
    raise RuntimeError(
        "No HF_TOKEN found. One-time setup: (1) accept the gate at "
        "huggingface.co/datasets/FlyRank/internship-warehouse (affiliation 'FlyRank ML Internship 2026'); "
        "(2) create a READ token (Settings -> Access Tokens, type Read); "
        "(3) make it available: locally run  setx HF_TOKEN \"hf_...\"  in a fresh terminal, "
        "or in Colab add a Secret named HF_TOKEN. Then re-run this notebook."
    )
print("token found: present in environment (value not printed)")

from huggingface_hub import snapshot_download

def find_repo_root():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "data" / "raw" / "content_refresh_anonymized.csv").exists():
            return candidate
    # Cloud-runtime fallback: the Colab badge opens just the notebook, not the repo.
    for parent in [Path("/content"), Path.cwd()]:
        target = parent / "flyrank-ml"
        if not target.parent.exists():
            continue
        if not (target / "data" / "raw" / "content_refresh_anonymized.csv").exists():
            print(f"repo not found locally; cloning the public repo into {target} ...")
            subprocess.run(["git", "clone", "--depth", "1", "https://github.com/tanay1122/flyrank-ml.git", str(target)], check=False, capture_output=True)
        if (target / "data" / "raw" / "content_refresh_anonymized.csv").exists():
            return target
    raise FileNotFoundError("No flyrank-ml repo found — run from inside the repo (or let the Colab fallback clone it).")

REPO = find_repo_root()
CACHE = REPO / "work" / "cache" / "warehouse"
LOCAL = snapshot_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    allow_patterns=["dim_clients.parquet", "fact_content_daily_performance/month=2026-03/*.parquet"],
    token=HF_TOKEN,
    local_dir=str(CACHE),
)
CACHE_DIR = Path(LOCAL)

FACT_MAR = (CACHE_DIR / "fact_content_daily_performance" / "month=2026-03" / "data_0.parquet").as_posix()
print("local files ready:")
print("  ", Path(FACT_MAR).name, "-", round(Path(FACT_MAR).stat().st_size / 1e6, 1), "MB (March partition)")

con = duckdb.connect()


[notice] A new release of pip is available: 26.0 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
duckdb 2.0.0
token found: present in environment (value not printed)


local files ready:
   data_0.parquet - 124.2 MB (March partition)


In [2]:
# This cell is for CODE (numbers, a query, a check).
# Build the frame (same policy as the lane's other notebooks), then distribution reads.
import sys

import numpy as np
import pandas as pd

pd.set_option("display.width", 200)

frame = con.execute(f'''
    WITH early AS (
        SELECT client_hash_id, content_hash_id, report_date,
               gsc_impressions, gsc_clicks, gsc_avg_position
        FROM read_parquet('{FACT_MAR}')
        WHERE report_date <= DATE '2026-03-20' AND gsc_data_available IS TRUE
    ), aggregated AS (
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_impressions) AS imp_early_tot,
               SUM(gsc_clicks) AS clicks_early_tot,
               AVG(CASE WHEN report_date >= DATE '2026-03-14' THEN gsc_impressions END) AS pace_mean,
               AVG(CASE WHEN report_date <= DATE '2026-03-13' THEN gsc_impressions END) AS base_mean,
               AVG(gsc_avg_position) AS pos_mean
        FROM early
        GROUP BY 1, 2
        HAVING SUM(gsc_impressions) >= 100 AND COUNT(*) >= 10
    )
    SELECT *,
           pace_mean / NULLIF(base_mean, 0) AS momentum,
           100.0 * clicks_early_tot / imp_early_tot AS click_rate_pct
    FROM aggregated
''').df()

target = con.execute(f'''
    SELECT client_hash_id, content_hash_id, AVG(gsc_impressions) AS target_mean
    FROM read_parquet('{FACT_MAR}')
    WHERE report_date >= DATE '2026-03-21' AND report_date <= DATE '2026-03-31'
          AND gsc_data_available IS TRUE
    GROUP BY 1, 2
''').df()

frame = frame.merge(target, on=["client_hash_id", "content_hash_id"], how="left")
no_outcome = int(frame["target_mean"].isna().sum())
frame = frame[frame["target_mean"].notna()].copy()
pace_ok = frame["pace_mean"] > 0
no_pace = int((~pace_ok).sum())
frame = frame[pace_ok].copy()
frame["decline_11d"] = (frame["target_mean"] < 0.70 * frame["pace_mean"]).astype(int)

print(f"Frame: {len(frame):,} pages | drops: {no_outcome:,} absent, {no_pace:,} no-pace | base rate {frame['decline_11d'].mean() * 100:.1f}%")
print()
dist = frame[["imp_early_tot", "click_rate_pct", "pos_mean", "momentum"]].describe(
    percentiles=[0.05, 0.25, 0.5, 0.75, 0.9, 0.99]
).T
dist["p99_over_p50"] = dist["99%"] / dist["50%"]
print("DISTRIBUTIONS (heavy-tail plan inputs):")
print(dist.to_string(formatters={c: "{:.3f}".format for c in dist.columns}))
print()
print(f"  p99/p50 receipts: imp {dist.loc['imp_early_tot', 'p99_over_p50']:.1f}x | momentum {dist.loc['momentum', 'p99_over_p50']:.2f}x  (imp is giant-tailed: buckets/ Spearman, not raw Pearson)")
pos_zero = int((frame["pos_mean"] <= 0).sum())
pos_null = int(frame["pos_mean"].isna().sum())
print(f"  position trap check: pos_mean == 0 rows: {pos_zero:,} | pos_mean NULL rows: {pos_null:,} (zero here means 'no position recorded', not rank #0)")

true_ctr = 100.0 * frame["clicks_early_tot"].sum() / frame["imp_early_tot"].sum()
mean_ctr = frame["click_rate_pct"].mean()
print(f"  CTR trap check: TRUE aggregate rate (total clicks / total imps) = {true_ctr:.2f}%  vs  mean of per-page CTRs = {mean_ctr:.2f}%")
print("  -> a rule or model reading the MEAN of per-page rates is listening to small pages too loudly.")

Frame: 83,258 pages | drops: 314 absent, 37 no-pace | base rate 15.9%

DISTRIBUTIONS (heavy-tail plan inputs):
                   count     mean      std     min      5%     25%     50%      75%      90%       99%        max p99_over_p50
imp_early_tot  83258.000 2007.405 4540.008 100.000 122.000 262.000 677.000 1890.000 4706.000 20370.000 244485.000       30.089
click_rate_pct 83258.000    0.277    0.442   0.000   0.000   0.000   0.132    0.398    0.736     1.942     18.750       14.667
pos_mean       83258.000   13.332   13.232   0.025   2.235   4.643   8.189   17.594   30.970    65.673    112.213        8.020
momentum       83258.000    1.101    1.314   0.000   0.333   0.667   0.921    1.237    1.724     4.501    133.333        4.885

  p99/p50 receipts: imp 30.1x | momentum 4.89x  (imp is giant-tailed: buckets/ Spearman, not raw Pearson)
  position trap check: pos_mean == 0 rows: 0 | pos_mean NULL rows: 0 (zero here means 'no position recorded', not rank #0)
  CTR trap check: TRUE a

## 2. Signal test #1 / #2 / #3 (verdict each)

Structure per the skill, identical for all three: **claim** in one sentence, **test** = grouped table with n's visible (floors: buckets under 50 rows flagged "insufficient"), **verdict** = CONFIRMED / OPPOSITE / MIXED / FALSE, judged by the same pre-registered criterion (ends ≥ 5pp apart, no opposing ≥ 5pp step). **One verdict convention for every test in this lane (same as the baseline notebook's checks): CONFIRMED always means "the decline rate RISES across the variable's ordered buckets"; OPPOSITE = it falls; MIXED = swings both ways; FALSE = flat within 5pp.** Continuous companions: Spearman rank correlations (heavy-tail-safe).

1. **Momentum vs decline** — the test that recalibrated this lane's whole baseline (the staircase shape was ALREADY read, disclosed, during baseline calibration in the sibling notebook `w04_baseline_score` §2; here it is the *robustness* story: a new decision date, a new slice, does the shape survive?).
2. **Volume vs decline** — pre-registered claim in convention terms: "risk rises with volume" (the 'big pages get hit harder' fear). Expectation going in: **OPPOSITE** — giants are safer (ML-05's single-feature AUC 0.392 receipt). Raw Pearson vs Spearman printed to show why the transform matters.
3. **Click rate vs decline** — pre-registered claim in convention terms: "risk rises with CTR" (pages that convert crash when demand moves — the anti-folklore reading; folklore says high CTR = healthy = safe, i.e. it predicts **OPPOSITE**). One caution up front: CTR is confounded with position and volume, so the isolated version is §3's within-band construction. Honest read, whatever comes out.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Test 1: momentum bands vs decline (main slice) + Spearman + the SECOND-SLICE rerun for robustness.
def verdict_ending(start_rate: float, end_rate: float, steps: list) -> str:
    delta = end_rate - start_rate
    any_down = min(steps) <= -5.0
    any_up = max(steps) >= 5.0
    if delta >= 5.0 and not any_down:
        return "CONFIRMED"
    if delta <= -5.0 and not any_up:
        return "OPPOSITE"
    if any_up and any_down:
        return "MIXED"
    if abs(delta) < 5.0:
        return "FALSE"
    return "MIXED"

mbands = [
    ("<=0.5 (deep freeze)", frame["momentum"] <= 0.5),
    ("0.5-0.7", frame["momentum"].between(0.5, 0.7, inclusive="right")),
    ("0.7-1.0", frame["momentum"].between(0.7, 1.0, inclusive="right")),
    ("1.0-1.3", frame["momentum"].between(1.0, 1.3, inclusive="right")),
    ("1.3-2.0", frame["momentum"].between(1.3, 2.0, inclusive="right")),
    (">2.0 (hot spike)", frame["momentum"] > 2.0),
]
tab1 = pd.DataFrame(
    [{"band": name, "n": int(m.sum()), "decline_pct": 100 * frame.loc[m, "decline_11d"].mean() if int(m.sum()) else float("nan")} for name, m in mbands]
)
print("TEST 1 (main slice) - claim: higher momentum (recent/early pace) -> higher decline risk")
print(tab1.to_string(index=False, formatters={"decline_pct": "{:.1f}".format}))
steps1 = tab1["decline_pct"].dropna().diff().dropna().tolist()
v1 = verdict_ending(float(tab1["decline_pct"].dropna().iloc[0]), float(tab1["decline_pct"].dropna().iloc[-1]), steps1)
s1 = frame["momentum"].corr(frame["decline_11d"], method="spearman")
print(f"  verdict (main slice): {v1} | Spearman(momentum, decline) = {s1:+.3f}")

print()
print("TEST 1 ROBUSTNESS RERUN - same construction, DIFFERENT slice: decision 03-10, features 03-01..10, target 03-11..17")
slice2 = con.execute(f'''
    WITH early2 AS (
        SELECT client_hash_id, content_hash_id, report_date, gsc_impressions
        FROM read_parquet('{FACT_MAR}')
        WHERE report_date <= DATE '2026-03-10' AND gsc_data_available IS TRUE
    ), agg2 AS (
        SELECT client_hash_id, content_hash_id,
               AVG(CASE WHEN report_date >= DATE '2026-03-04' THEN gsc_impressions END) AS pace_mean,
               AVG(CASE WHEN report_date <= DATE '2026-03-03' THEN gsc_impressions END) AS base_mean
        FROM early2
        GROUP BY 1, 2
        HAVING SUM(gsc_impressions) >= 100 AND COUNT(*) >= 10
    )
    SELECT *,
           pace_mean / NULLIF(base_mean, 0) AS momentum
    FROM agg2
''').df()
tgt2 = con.execute(f'''
    SELECT client_hash_id, content_hash_id, AVG(gsc_impressions) AS target_mean
    FROM read_parquet('{FACT_MAR}')
    WHERE report_date >= DATE '2026-03-11' AND report_date <= DATE '2026-03-17'
          AND gsc_data_available IS TRUE
    GROUP BY 1, 2
''').df()
slice2 = slice2.merge(tgt2, on=["client_hash_id", "content_hash_id"], how="left")
n_absent2 = int(slice2["target_mean"].isna().sum())
slice2 = slice2[slice2["target_mean"].notna() & (slice2["pace_mean"] > 0)].copy()
slice2["decline_w"] = (slice2["target_mean"] < 0.70 * slice2["pace_mean"]).astype(int)
print(f"  slice-2: {len(slice2):,} pages kept ({n_absent2:,} absent from target window) | base rate {slice2['decline_w'].mean() * 100:.1f}%")
mbands2 = [
    ("<=0.5", slice2["momentum"] <= 0.5),
    ("0.5-1.0", slice2["momentum"].between(0.5, 1.0, inclusive="right")),
    ("1.0-1.3", slice2["momentum"].between(1.0, 1.3, inclusive="right")),
    ("1.3-2.0", slice2["momentum"].between(1.3, 2.0, inclusive="right")),
    (">2.0", slice2["momentum"] > 2.0),
]
tab1b = pd.DataFrame(
    [{"band": name, "n": int(m.sum()), "decline_pct": 100 * slice2.loc[m, "decline_w"].mean() if int(m.sum()) else float("nan")} for name, m in mbands2]
)
print(tab1b.to_string(index=False, formatters={"decline_pct": "{:.1f}".format}))
print(f"  momentum NULL in slice 2 (excluded from bands): {int(slice2['momentum'].isna().sum()):,}")
steps1b = tab1b["decline_pct"].dropna().diff().dropna().tolist()
v1b = verdict_ending(float(tab1b["decline_pct"].dropna().iloc[0]), float(tab1b["decline_pct"].dropna().iloc[-1]), steps1b)
s1b = slice2["momentum"].corr(slice2["decline_w"], method="spearman")
print(f"  verdict (slice 2): {v1b} | Spearman = {s1b:+.3f}")
print(f"  Survival receipt: main {v1}, slice-2 {v1b} -> {'signal SURVIVES the new slice' if v1 == v1b == 'CONFIRMED' else 'shape does not fully survive - noted honestly'}")

TEST 1 (main slice) - claim: higher momentum (recent/early pace) -> higher decline risk
               band     n decline_pct
<=0.5 (deep freeze) 10468        14.7
            0.5-0.7 12848        12.3
            0.7-1.0 24727        12.3
            1.0-1.3 17067        15.2
            1.3-2.0 12385        22.8
   >2.0 (hot spike)  5763        29.8


  verdict (main slice): CONFIRMED | Spearman(momentum, decline) = +0.107

TEST 1 ROBUSTNESS RERUN - same construction, DIFFERENT slice: decision 03-10, features 03-01..10, target 03-11..17


  slice-2: 56,229 pages kept (39 absent from target window) | base rate 30.1%
   band     n decline_pct
  <=0.5  2825        33.3
0.5-1.0 23072        27.8
1.0-1.3 14607        31.0
1.3-2.0 10778        30.2
   >2.0  4947        35.9
  momentum NULL in slice 2 (excluded from bands): 0
  verdict (slice 2): MIXED | Spearman = +0.025
  Survival receipt: main CONFIRMED, slice-2 MIXED -> shape does not fully survive - noted honestly


In [4]:
# This cell is for CODE (numbers, a query, a check).
# Test 2: volume vs decline - raw Pearson vs Spearman on log1p (heavy-tail receipts), then buckets.
print("TEST 2 - claim (risk-gradient convention, per section 2): risk rises with volume ('big pages get hit harder');")
print("  expectation going in: OPPOSITE (giants are safer, per the ML-05 single-feature receipt)")
p_raw = frame["imp_early_tot"].corr(frame["decline_11d"].astype(float), method="pearson")
p_rank = frame["imp_early_tot"].rank().corr(frame["decline_11d"], method="spearman")
p_logp1 = np.log1p(frame["imp_early_tot"]).corr(frame["decline_11d"].astype(float), method="pearson")
print(f"  raw Pearson(imp, decline) = {p_raw:+.4f}   (giant-dominated)")
print(f"  Spearman(imp, decline)    = {p_rank:+.4f}   (rank-based, heavy-tail-safe)")
print(f"  Pearson on log1p(imp)     = {p_logp1:+.4f}   (log tames the tail)")

import numpy as np
frame["volume_bucket"] = pd.cut(
    frame["imp_early_tot"],
    bins=[-1, 500, 2000, 10000, 50000, np.inf],
    labels=["100-500 (near floor)", "500-2k", "2k-10k", "10k-50k", "50k+ (giants)"],
)
tab2 = (
    frame.groupby("volume_bucket", observed=True)
    .agg(n=("decline_11d", "size"), decline_pct=("decline_11d", lambda s: 100 * s.mean()))
    .reset_index()
)
print(tab2.to_string(index=False, formatters={"decline_pct": "{:.1f}".format}))
steps2 = tab2["decline_pct"].diff().dropna().tolist()
v2 = verdict_ending(float(tab2["decline_pct"].iloc[0]), float(tab2["decline_pct"].iloc[-1]), steps2)
print(f"  verdict: {v2}  <- small pages decline at ~2x the giants' rate; volume is NOT a risk axis here, it is the prize axis")

TEST 2 - claim (risk-gradient convention, per section 2): risk rises with volume ('big pages get hit harder');
  expectation going in: OPPOSITE (giants are safer, per the ML-05 single-feature receipt)
  raw Pearson(imp, decline) = -0.0804   (giant-dominated)
  Spearman(imp, decline)    = -0.1364   (rank-based, heavy-tail-safe)
  Pearson on log1p(imp)     = -0.1366   (log tames the tail)
       volume_bucket     n decline_pct
100-500 (near floor) 35073        20.8
              500-2k 28374        15.0
              2k-10k 16872         9.0
             10k-50k  2844         6.3
       50k+ (giants)    95         7.4
  verdict: OPPOSITE  <- small pages decline at ~2x the giants' rate; volume is NOT a risk axis here, it is the prize axis


In [5]:
# This cell is for CODE (numbers, a query, a check).
# Test 3: click rate vs decline - quantile buckets (handles the heavy tail of the rate itself), verdict; floors respected.
print("TEST 3 - claim (risk-gradient convention, per section 2): risk rises with CTR (pages that convert crash);")
print("  the folklore 'high CTR = healthy engagement = safe' predicts the OPPOSITE")
frame["ctr_band"] = pd.qcut(
    frame["click_rate_pct"].rank(method="first"),  # rank-first tiers: ties (many identical low CTRs) split deterministically
    q=4,
    labels=["Q1 (lowest CTR)", "Q2", "Q3", "Q4 (highest CTR)"],
)
tab3 = (
    frame.groupby("ctr_band", observed=True)
    .agg(n=("decline_11d", "size"), decline_pct=("decline_11d", lambda s: 100 * s.mean()), median_imp=("imp_early_tot", "median"))
    .reset_index()
)
print(tab3.to_string(index=False, formatters={"decline_pct": "{:.1f}".format, "median_imp": "{:.0f}".format}))
steps3 = tab3["decline_pct"].diff().dropna().tolist()
v3 = verdict_ending(float(tab3["decline_pct"].iloc[0]), float(tab3["decline_pct"].iloc[-1]), steps3)
s3 = frame["click_rate_pct"].rank().corr(frame["decline_11d"], method="spearman")
small = tab3[tab3["n"] < 50]["n"].tolist()
print(f"  verdict: {v3} = decline falls as CTR rises (Q1 21.9% -> Q4 11.0%): the folklore direction wins the RAW read")
print(f"  Spearman(click_rate, decline) = {s3:+.3f} | floors: {'all buckets >= 50' if not small else f'bucket(s) under the 50-row floor: {small}'}")
print("  caution receipt: CTR inherits its raw shape from volume (median imp ~269 in Q1 vs ~832 in Q4) -")
print("  the honest isolation is the within-position-band test in section 3, before any verdict about the flag.")

TEST 3 - claim (risk-gradient convention, per section 2): risk rises with CTR (pages that convert crash);
  the folklore 'high CTR = healthy engagement = safe' predicts the OPPOSITE
        ctr_band     n decline_pct median_imp
 Q1 (lowest CTR) 20815        21.9        272
              Q2 20814        17.9        879
              Q3 20814        13.0       1332
Q4 (highest CTR) 20815        11.0        832
  verdict: OPPOSITE = decline falls as CTR rises (Q1 21.9% -> Q4 11.0%): the folklore direction wins the RAW read
  Spearman(click_rate, decline) = -0.122 | floors: all buckets >= 50
  caution receipt: CTR inherits its raw shape from volume (median imp ~269 in Q1 vs ~832 in Q4) -
  the honest isolation is the within-position-band test in section 3, before any verdict about the flag.


## 3. The flag-linked test

**The CTR-fix logic, audited at its own horizon.** A FlyRank flag ranks pages whose *position is good but CTR is poor* (title/snippet leaving clicks on the table) for an indexed fix. Its embedded assumption in THIS lane's terms: a page failing to convert its position into clicks is also a page whose traffic sits on a weak base — so when demand moves, it should decline MORE than its high-CTR-at-same-position peers. Test: within each position band, compare decline rates of pages BELOW vs ABOVE that band's OWN median CTR (band-relative, because CTR falls with position — a raw CTR cut would hide the effect). Floors respected: cross-cut cells need n ≥ 30; smaller cells are printed as "insufficient", not read. Verdict may be a negative — that is a finding, not a failure.

Position = `pos_mean` over kept pages (0-or-NULL receipt printed in §1; those rows do not exist here).

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Flag-linked test: position band x band-relative CTR tier -> decline rate; floors and verdict printed.
import numpy as np

frame["pos_band"] = pd.cut(frame["pos_mean"], bins=[-1, 3, 10, 30, np.inf], labels=["top-3", "4-10", "11-30", "31+"])

cells = []
for band in ["top-3", "4-10", "11-30", "31+"]:
    sub = frame[frame["pos_band"] == band]
    if len(sub) < 30:
        cells.append({"position": band, "n": len(sub), "ctr_tier": "insufficient data", "decline_pct": float("nan")})
        continue
    med = sub["click_rate_pct"].median()
    for tier, mask in [("low CTR", sub["click_rate_pct"] < med), ("high CTR", sub["click_rate_pct"] >= med)]:
        s = sub[mask]
        cells.append({"position": band, "n": len(s), "ctr_tier": tier, "decline_pct": 100 * s["decline_11d"].mean()})

tab4 = pd.DataFrame(cells)
print("FLAG-LINKED TEST - CTR-fix set: within each position band, does BELOW-median CTR precede MORE decline?")
print(tab4.to_string(index=False, formatters={"decline_pct": "{:.1f}".format}))
print()
verdict_cells = []
for band in ["top-3", "4-10", "11-30", "31+"]:
    two = tab4[(tab4["position"] == band) & (tab4["ctr_tier"].isin(["low CTR", "high CTR"]))]
    if len(two) == 2 and (two["n"] >= 30).all():
        gap = float(two[two["ctr_tier"] == "low CTR"]["decline_pct"].iloc[0] - two[two["ctr_tier"] == "high CTR"]["decline_pct"].iloc[0])
        verdict_cells.append({"position": band, "low_minus_high_gap_pp": round(gap, 1)})
    else:
        verdict_cells.append({"position": band, "low_minus_high_gap_pp": None})
print("gaps (low-CTR decline minus high-CTR decline, pp; None = insufficient):")
print(pd.DataFrame(verdict_cells).to_string(index=False))
gaps = [c["low_minus_high_gap_pp"] for c in verdict_cells if c["low_minus_high_gap_pp"] is not None]
if gaps:
    if all(g > 0 for g in gaps):
        v4 = "CONFIRMED (low-CTR pages decline more, at every readable position)"
    elif all(g < 0 for g in gaps):
        v4 = "OPPOSITE (low-CTR pages decline LESS)"
    else:
        v4 = f"MIXED (gaps flip sign across bands: {gaps})"
else:
    v4 = "FALSE/insufficient"
print(f"verdict: {v4}")
if v4.startswith("CONFIRMED"):
    print("Reading: the flag's embedded risk hypothesis HOLDS in this slice at the 11-day horizon -")
    print("below-median band-relative CTR preceded MORE decline at every readable position. Caveats: single")
    print("slice, single horizon; the 31+ band had no low-CTR cell above the floor. The Week-5 model may reuse")
    print("the same construction for a second opinion.")
else:
    print("Reading: this horizon is 11 days. A flag that pays over months can look different at 11 days -")
    print("the audit declines to confirm the flag AT THIS HORIZON, which the Week-5 model may revisit with")
    print("the same construction.")

FLAG-LINKED TEST - CTR-fix set: within each position band, does BELOW-median CTR precede MORE decline?
position     n ctr_tier decline_pct
   top-3  4367  low CTR        21.0
   top-3  4368 high CTR        10.0
    4-10 19549  low CTR        19.7
    4-10 19549 high CTR        11.9
   11-30 13221  low CTR        18.5
   11-30 13223 high CTR        11.2
     31+     0  low CTR         NaN
     31+  8981 high CTR        20.2

gaps (low-CTR decline minus high-CTR decline, pp; None = insufficient):
position  low_minus_high_gap_pp
   top-3                   11.0
    4-10                    7.7
   11-30                    7.3
     31+                    NaN
verdict: CONFIRMED (low-CTR pages decline more, at every readable position)
Reading: the flag's embedded risk hypothesis HOLDS in this slice at the 11-day horizon -
below-median band-relative CTR preceded MORE decline at every readable position. Caveats: single
slice, single horizon; the 31+ band had no low-CTR cell above the floor. The W

## 4. What this means in practice

Printed one line per test below, then the receipt JSON. The short version for a content team on this lane:

1. **Watch the spikes; treat the smooth ladder as slice-fragile.** In the main slice, decline risk climbed with momentum (12.3% in 0.7–1.0x -> 29.8% above 2x, CONFIRMED, Spearman +0.107). The rerun on a different decision date only PARTIALLY reproduced it (verdict MIXED, Spearman +0.025): the >2x hot-spike tier stayed the clearest above-base signal (35.9% vs 30.1% base), but the middle of the ladder flattened. Honest read: hot spikes are the durable core finding — a page that blew past its own early-month base is the priority queue item; the lower tiers are not reliable enough on their own to act on.
2. **Volume is the prize, not the danger.** Bigger pages decline *less* (OPPOSITE — clear in both the bands and Spearman −0.136). Ranking rules should spend reviewer slots on big pages from the risk side, and should absolutely not treat "high traffic" as a safety blanket either — the giants' 6–7% rate is not zero.
3. **The refresh-flag staleness story is inert at this floor** (the companion baseline notebook's §1: kept pages are near-daily visible, ~81% were rewritten after the decision date) — staleness points should stay a tie-break at most.
4. **The CTR-fix logic CONFIRMED in this slice, at the 11-day horizon.** Within every readable position band, below-median band-relative CTR preceded MORE decline (+11.0 / +7.7 / +7.3 pp at top-3 / 4–10 / 11–30). Treat as decision-support, not proof: one slice, one horizon, and the 31+ band had no low-CTR cell above the floor — the Week-5 model may reuse the same construction for a second opinion.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Verdict summary + receipt JSON (aggregates only, safe to commit).
import json

summary = pd.DataFrame([
    {"signal": "momentum (recent/early pace)", "claimed": "risk rises with momentum", "verdict": v1, "slice_2": v1b, "spearman": round(s1, 3)},
    {"signal": "volume (impressions)", "claimed": "risk rises with volume (big = fragile)", "verdict": v2, "slice_2": "-", "spearman": round(p_rank, 3)},
    {"signal": "click rate (CTR)", "claimed": "risk rises with CTR (converting pages crash)", "verdict": v3, "slice_2": "-", "spearman": round(s3, 3)},
    {"signal": "CTR-fix flag (position x CTR)", "claimed": "low CTR at good position -> more decline", "verdict": v4.split(" ")[0] if " " in v4 else v4, "slice_2": "-", "spearman": "-"},
])
print("VERDICT SUMMARY (every verdict rests on tables with n's printed above; no bucket under the floor was read):")
print(summary.to_string(index=False))

OUT_DIR = REPO / "work" / "outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)
receipts = {
    "notebook": "w04_signal_audit (ML-06, optional full-depth sibling of the ML-07 card)",
    "slice": "month=2026-03, one row = one page, floor 100 imps / 10 days, gsc IS TRUE, aggregates <= 03-20",
    "frame_pages": int(len(frame)),
    "base_rate_pct": round(float(frame["decline_11d"].mean()) * 100, 1),
    "test1_momentum": {"main": v1, "slice2": v1b, "spearman_main": round(s1, 3), "spearman_slice2": round(s1b, 3),
                       "table_main": json.loads(tab1.to_json(orient="records", date_format="iso")),
                       "table_slice2": json.loads(tab1b.to_json(orient="records", date_format="iso"))},
    "test2_volume": {"verdict": v2, "spearman": round(p_rank, 3), "pearson_raw": round(p_raw, 4), "pearson_log1p": round(p_logp1, 4),
                     "table": json.loads(tab2.to_json(orient="records", date_format="iso"))},
    "test3_ctr": {"verdict": v3, "spearman": round(s3, 3), "table": json.loads(tab3.to_json(orient="records", date_format="iso"))},
    "flag_test_ctrfix": {"verdict": v4, "cells": json.loads(tab4.to_json(orient="records", date_format="iso")), "gaps": verdict_cells},
    "ctr_trap": {"true_aggregate_pct": round(float(true_ctr), 2), "mean_of_page_ctrs_pct": round(float(mean_ctr), 2)},
    "position_trap": {"pos_mean_zero_rows": pos_zero, "pos_mean_null_rows": pos_null},
    "lane_lock": "Lane 2 (Refresh / Content Opportunity Scoring) CONFIRMED + LOCKED, week 4",
}
with open(OUT_DIR / "w04_signal_receipt.json", "w", encoding="utf-8") as handle:
    json.dump(receipts, handle, indent=2)
print("receipt JSON written: work/outputs/w04_signal_receipt.json (aggregates only - safe to commit)")

VERDICT SUMMARY (every verdict rests on tables with n's printed above; no bucket under the floor was read):
                       signal                                      claimed   verdict slice_2 spearman
 momentum (recent/early pace)                     risk rises with momentum CONFIRMED   MIXED    0.107
         volume (impressions)       risk rises with volume (big = fragile)  OPPOSITE       -   -0.136
             click rate (CTR) risk rises with CTR (converting pages crash)  OPPOSITE       -   -0.122
CTR-fix flag (position x CTR)     low CTR at good position -> more decline CONFIRMED       -        -
receipt JSON written: work/outputs/w04_signal_receipt.json (aggregates only - safe to commit)


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.